# Results for model: meta/llama-3.3-70b-instruct

In [1]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Convert date_id to datetime
df = df.with_columns(pl.col('date_id').cast(pl.Int64).alias('date_id'))

# Calculate global TOP_QUANTILE
df = df.with_columns(
    pl.col('feature_00').rank(method='dense').alias('feature_00_rank')
)
df = df.with_columns(
    pl.when(pl.col('feature_00_rank') > (pl.col('feature_00_rank').max() * 0.85)).then(1).otherwise(0).alias('TOP_QUANTILE')
)

# Define rolling window size
window_size = 30

# Calculate rolling TOP_QUANTILE
df = df.with_columns(
    pl.col('date_id').alias('date_id_sort')
)
df = df.sort('date_id_sort')
df = df.with_columns(
    pl.col('TOP_QUANTILE').rolling(window_size).mean().alias('rolling_TOP_QUANTILE')
)

# Prepare data for training
X = df.drop(['responder_6', 'date_id_sort', 'TOP_QUANTILE']).to_pandas()
y = df['responder_6'].to_pandas()

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost Regressor
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', max_depth=5, learning_rate=0.1, n_estimators=100, n_jobs=-1)
xgb_model.fit(X_train, y_train)

# Make predictions
y_pred = xgb_model.predict(X_test)

# Evaluate model
mse = mean_squared_error(y_test, y_pred)
print(f'MSE: {mse}')

# Train on full dataset
xgb_model.fit(X, y)

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet